In [4]:
from pathlib import Path
from collections import defaultdict
from typing import Callable

import math
import re
import sys
import unicodedata
import xml.etree.ElementTree as ET

import pandas as pd

from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

### Leemos los documentos

In [5]:
DOCS_PATH = Path("docs-raw-texts")
QUERIES_PATH = Path("queries-raw-texts")
RELEVANCE_PATH = Path("relevance-judgments.tsv")

In [6]:
def read_naf(path: Path) -> str:
    root = ET.parse(path).getroot()

    title = ""
    raw = ""

    for elem in root.iter():
        tag = elem.tag.split("}")[-1]

        if tag == "fileDesc":
            title = elem.attrib.get("title", "")

        elif tag == "raw":
            raw = elem.text or ""

    return f"{title} {raw}".strip()

def get_id(path: Path) -> str:
    return path.stem.replace("wes2015.", "")

In [7]:
docs = {
    get_id(path): read_naf(path)
    for path in DOCS_PATH.glob("*.naf")
}

queries = {
    get_id(path): read_naf(path)
    for path in QUERIES_PATH.glob("*.naf")
}

print("Documentos:", len(docs))
print("Queries:", len(queries))

Documentos: 331
Queries: 35


Veamos un ejemplo

In [9]:
doc_id = next(iter(docs))

print(doc_id)
print(docs[doc_id][:1000])

queries_id = next(iter(queries))


print(queries_id)
print(queries[queries_id][:1000])

d001
William Beaumont and the Human Digestion William Beaumont and the Human Digestion.

William Beaumont: Physiology of digestion Image Source.  On November 21, 1785, US-American surgeon William Beaumont was born. He became best known as “Father of Gastric Physiology” following his research on human digestion. William Beaumont was born in Lebanon, Connecticut and became a physician. He served as a surgeon’s mate in the Army during the War of 1812. He opened a private practice in Plattsburgh, New York, but rejoined the Army as a surgeon in 1819. Beaumont was stationed at Fort Mackinac on Mackinac Island in Michigan in the early 1820s when it existed to protect the interests of the American Fur Company. The fort became the refuge for a wounded 19-year-old French-Canadian fur trader named Alexis St. Martin when a shotgun went off by accident in the American Fur Company store at close range June 6th, 1822. St. Martin’s wound was quite serious because his stomach was perforated and several

### Tokenización

Que vamos a conserver: números, fechas como 2020-10-25, mantener palabras con guion como un único token: state-of-the-art, conservar inicialmente caracteres acentuados.

In [10]:
TOKEN_PATTERN = re.compile(
    r"\d{1,4}(?:[/-]\d{1,2}){1,2}"   # fechas
    r"|\d+(?:[.,]\d+)?"              # números
    r"|[^\W\d_]+(?:-[^\W\d_]+)*",    # palabras
    re.UNICODE,
)

In [11]:
def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(text)

In [12]:
tokenize("The state-of-the-art model cost 25.5 dollars in 2024-05-10.")

['The',
 'state-of-the-art',
 'model',
 'cost',
 '25.5',
 'dollars',
 'in',
 '2024-05-10']

### Stopwords

In [13]:
STOPWORDS = set(ENGLISH_STOP_WORDS)

In [14]:
def remove_stopwords(tokens: list[str]) -> list[str]:
    return [
        token
        for token in tokens
        if token.lower() not in STOPWORDS
    ]

### Normalizacion

1. Convertir a minúsculas.
2. Quitar tildes/acentos.

In [15]:
def normalize_token(token: str) -> str:
    token = token.lower()

    token = unicodedata.normalize("NFKD", token)

    return "".join(
        char
        for char in token
        if not unicodedata.combining(char)
    )

In [16]:
def normalize(tokens: list[str]) -> list[str]:
    return [normalize_token(token) for token in tokens]

### Stemming

In [17]:
stemmer = SnowballStemmer("english")

In [18]:
def stem_token(token: str) -> str:
    if any(char.isdigit() for char in token):
        return token

    return "-".join(
        stemmer.stem(part)
        for part in token.split("-")
    )

def stem(tokens: list[str]) -> list[str]:
    return [stem_token(token) for token in tokens]

## Pipeline

In [19]:
def preprocess(text: str) -> list[str]:
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    tokens = normalize(tokens)
    tokens = stem(tokens)

    return tokens

In [21]:
### Testeo
text = "The computers are running state-of-the-art algorithms."

print(preprocess(text))

['comput', 'run', 'state-of-the-art', 'algorithm']


In [22]:
def preprocessing_steps(text: str) -> dict[str, list[str]]:
    tokenized = tokenize(text)
    no_stopwords = remove_stopwords(tokenized)
    normalized = normalize(no_stopwords)
    stemmed = stem(normalized)

    return {
        "Tokenización": tokenized,
        "Stopwords": no_stopwords,
        "Normalización": normalized,
        "Stemming": stemmed,
    }

### Tamaño del vocabulario - número total de tokens

In [23]:
def collection_stats(
    collection: dict[str, str],
    name: str,
) -> list[dict]:
    
    stages: dict[str, list[str]] = defaultdict(list)

    for text in collection.values():
        result = preprocessing_steps(text)

        for stage, tokens in result.items():
            stages[stage].extend(tokens)

    return [
        {
            "Colección": name,
            "Paso": stage,
            "Tokens": len(tokens),
            "Vocabulario": len(set(tokens)),
        }
        for stage, tokens in stages.items()
    ]

In [24]:
stats = (
    collection_stats(docs, "Documentos")
    + collection_stats(queries, "Queries")
)

stats_df = pd.DataFrame(stats)

stats_df

,Colección,Paso,Tokens,Vocabulario
0,Documentos,Tokenización,216227,21729
1,Documentos,Stopwords,116556,21255
2,Documentos,Normalización,116556,19086
3,Documentos,Stemming,116556,13994
4,Queries,Tokenización,156,126
5,Queries,Stopwords,116,107
6,Queries,Normalización,116,104
7,Queries,Stemming,116,100


### Documentos procesados

In [25]:
docs_processed = {
    doc_id: preprocess(text)
    for doc_id, text in docs.items()
}

queries_processed = {
    query_id: preprocess(text)
    for query_id, text in queries.items()
}

In [26]:
doc_id = next(iter(docs_processed))

print(doc_id)
print(docs_processed[doc_id][:50])

d001
['william', 'beaumont', 'human', 'digest', 'william', 'beaumont', 'human', 'digest', 'william', 'beaumont', 'physiolog', 'digest', 'imag', 'sourc', 'novemb', '21', '1785', 'us-american', 'surgeon', 'william', 'beaumont', 'born', 'best', 'known', 'father', 'gastric', 'physiolog', 'follow', 'research', 'human', 'digest', 'william', 'beaumont', 'born', 'lebanon', 'connecticut', 'physician', 'serv', 'surgeon', 's', 'mate', 'armi', 'war', '1812', 'open', 'privat', 'practic', 'plattsburgh', 'new', 'york']


# Indice invertido

In [27]:
def build_inverted_index(
    documents: dict[str, list[str]],
) -> dict[str, list[str]]:
    
    index = defaultdict(set)

    for doc_id, tokens in documents.items():
        for token in tokens:
            index[token].add(doc_id)

    return {
        term: sorted(postings)
        for term, postings in index.items()
    }

In [28]:
inverted_index = build_inverted_index(docs_processed)

print("Términos:", len(inverted_index))

Términos: 13994


In [32]:
term = next(iter(inverted_index))

print(term)
print(inverted_index[term])
print(len(inverted_index[term]))

william
['d001', 'd009', 'd015', 'd028', 'd035', 'd055', 'd056', 'd069', 'd078', 'd088', 'd091', 'd092', 'd095', 'd098', 'd102', 'd106', 'd109', 'd111', 'd129', 'd136', 'd138', 'd147', 'd175', 'd179', 'd180', 'd189', 'd190', 'd191', 'd193', 'd197', 'd212', 'd230', 'd231', 'd241', 'd254', 'd257', 'd266', 'd272', 'd273', 'd274', 'd289', 'd291', 'd294', 'd299', 'd300', 'd309', 'd310', 'd320', 'd323', 'd330']
50


### Posting mas frecuente y menos frecuente

In [33]:
most_frequent = max(
    inverted_index,
    key=lambda term: len(inverted_index[term]),
)

least_frequent = min(
    inverted_index,
    key=lambda term: len(inverted_index[term]),
)

In [34]:
print(
    "Más frecuente:",
    most_frequent,
    len(inverted_index[most_frequent]),
)

print(
    "Menos frecuente:",
    least_frequent,
    len(inverted_index[least_frequent]),
)

Más frecuente: s 321
Menos frecuente: beaumont 1


### Skip pointers

In [35]:
def build_skips(postings: list[str]) -> dict[int, int]:
    p = len(postings)

    if p < 4:
        return {}

    step = int(math.sqrt(p))

    return {
        i: i + step
        for i in range(0, p - step, step)
    }

In [ ]:
# testing
postings = list(range(16))

build_skips(postings)

{0: 4, 4: 8, 8: 12}

In [37]:
skip_index = {
    term: build_skips(postings)
    for term, postings in inverted_index.items()
}

In [38]:
print(inverted_index[most_frequent][:20])
print(skip_index[most_frequent])

['d001', 'd002', 'd003', 'd004', 'd005', 'd006', 'd007', 'd009', 'd010', 'd011', 'd012', 'd013', 'd014', 'd015', 'd016', 'd017', 'd018', 'd019', 'd020', 'd021']
{0: 17, 17: 34, 34: 51, 51: 68, 68: 85, 85: 102, 102: 119, 119: 136, 136: 153, 153: 170, 170: 187, 187: 204, 204: 221, 221: 238, 238: 255, 255: 272, 272: 289, 289: 306}


In [39]:
def doc_number(doc_id: str) -> int:
    return int(re.search(r"\d+", doc_id).group())

In [40]:
def intersect(
    p1: list[str],
    p2: list[str],
    s1: dict[int, int],
    s2: dict[int, int],
) -> list[str]:

    result = []

    i = 0
    j = 0

    while i < len(p1) and j < len(p2):

        d1 = doc_number(p1[i])
        d2 = doc_number(p2[j])

        if d1 == d2:
            result.append(p1[i])
            i += 1
            j += 1

        elif d1 < d2:

            if i in s1 and doc_number(p1[s1[i]]) <= d2:
                i = s1[i]
            else:
                i += 1

        else:

            if j in s2 and doc_number(p2[s2[j]]) <= d1:
                j = s2[j]
            else:
                j += 1

    return result

In [41]:
term1 = "comput"
term2 = "model"

result = intersect(
    inverted_index.get(term1, []),
    inverted_index.get(term2, []),
    skip_index.get(term1, {}),
    skip_index.get(term2, {}),
)

result

['d028',
 'd116',
 'd129',
 'd177',
 'd193',
 'd194',
 'd195',
 'd196',
 'd197',
 'd198',
 'd220',
 'd222',
 'd266',
 'd286']

### Tamaño del índice en memoria

In [42]:
def deep_size(obj, seen=None) -> int:
    if seen is None:
        seen = set()

    if id(obj) in seen:
        return 0

    seen.add(id(obj))
    size = sys.getsizeof(obj)

    if isinstance(obj, dict):
        size += sum(
            deep_size(k, seen) + deep_size(v, seen)
            for k, v in obj.items()
        )

    elif isinstance(obj, (list, tuple, set)):
        size += sum(deep_size(x, seen) for x in obj)

    return size

In [43]:
index_size = deep_size(inverted_index)
index_with_skips_size = deep_size(
    (inverted_index, skip_index)
)

print(f"Índice: {index_size / 1024:.2f} KB")
print(f"Índice + skips: {index_with_skips_size / 1024:.2f} KB")

Índice: 2798.30 KB
Índice + skips: 4693.99 KB
